# EDA -- bronze

What columns each table in `data/bronze/` has as it arrives from the API, before any cleaning. No full dumps -- maximum 5 sample rows per table. Compare with `eda_silver.ipynb` to see what changes column by column at each bronze -> silver step.

In [1]:
import glob
import os
from pathlib import Path

import duckdb
import pandas as pd


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


os.chdir(_repo_root(Path.cwd()))

AIRE_PATH = "data/bronze/aire/*.parquet"
ESTACIONES_AIRE_PATH = "data/bronze/estaciones_aire/latest.parquet"
TRAFICO_PATH = "data/bronze/trafico/*.parquet"
DISTRITOS_PATH = "data/bronze/distritos/latest.parquet"
TRAFICO_PUNTOS_PATH = "data/bronze/trafico_puntos_medida/*.parquet"

## `aire`

Wide format as delivered by the API: numeric `MAGNITUD` code and 31 columns of data/validity per day (`D01..D31`, `V01..V31`), with `ANO`/`MES` instead of a `fecha` column. `unpivot_air_quality` (`src/data/silver/aire.py`) converts it to long format in silver.

In [2]:
aire = pd.read_parquet(sorted(glob.glob(AIRE_PATH)))
aire.shape

(2977, 69)

In [3]:
aire.dtypes

PROVINCIA           int64
MUNICIPIO          object
ESTACION            int64
MAGNITUD            int64
PUNTO_MUESTREO     object
                   ...   
V29                object
D30               float64
V30                object
D31               float64
V31                object
Length: 69, dtype: object

In [4]:
aire.head(5)

,PROVINCIA,MUNICIPIO,ESTACION,MAGNITUD,PUNTO_MUESTREO,ANO,MES,D01,V01,D02,...,D27,V27,D28,V28,D29,V29,D30,V30,D31,V31
0,28,079,4,1,28079004_1_38,2023,01,1.0,V,1.0,...,2.0,V,2.0,V,1.0,V,2.0,V,4.0,V
1,28,079,4,1,28079004_1_38,2023,02,3.0,V,4.0,...,2.0,V,2.0,V,0.0,N,0.0,N,0.0,N
2,28,079,4,1,28079004_1_38,2023,03,3.0,V,2.0,...,1.0,V,1.0,V,1.0,V,1.0,V,1.0,V
3,28,079,4,1,28079004_1_38,2023,04,1.0,V,1.0,...,2.0,V,1.0,V,1.0,V,1.0,V,0.0,N
4,28,079,4,1,28079004_1_38,2023,05,1.0,V,1.0,...,1.0,V,1.0,V,1.0,V,1.0,V,1.0,V


In [5]:
aire.isna().sum()

PROVINCIA         0
MUNICIPIO         0
ESTACION          0
MAGNITUD          0
PUNTO_MUESTREO    0
                 ..
V29               0
D30               0
V30               0
D31               0
V31               0
Length: 69, dtype: int64

## `estaciones_aire`

No `COD_DIS`/`NOMBRE` district columns yet -- those two columns are added by the spatial join in `assign_district` (`src/data/silver/district_join.py`) in silver.

In [6]:
estaciones_aire = pd.read_parquet(ESTACIONES_AIRE_PATH)
estaciones_aire.shape

(24, 25)

In [7]:
estaciones_aire.dtypes

CODIGO                   int64
CODIGO_CORTO             int64
ESTACION                object
DIRECCION               object
LONGITUD_ETRS89         object
LATITUD_ETRS89          object
ALTITUD                  int64
COD_TIPO                object
NOM_TIPO                object
NO2                     object
SO2                     object
CO                      object
PM10                    object
PM2_5                   object
O3                      object
BTX                     object
COD_VIA                  int64
VIA_CLASE               object
VIA_PAR                 object
VIA_NOMBRE              object
Fecha alta              object
COORDENADA_X_ETRS89     object
COORDENADA_Y_ETRS89     object
LONGITUD               float64
LATITUD                float64
dtype: object

In [9]:
estaciones_aire.head(5)

,CODIGO,CODIGO_CORTO,ESTACION,DIRECCION,LONGITUD_ETRS89,LATITUD_ETRS89,ALTITUD,COD_TIPO,NOM_TIPO,NO2,...,BTX,COD_VIA,VIA_CLASE,VIA_PAR,VIA_NOMBRE,Fecha alta,COORDENADA_X_ETRS89,COORDENADA_Y_ETRS89,LONGITUD,LATITUD
0,28079004,4,Plaza de España,Plaza de España,"3°42'43.91""O","40°25'25.98""N",637,UT,Urbana tráfico,X,...,None,273600,PLAZA,DE,ESPAÑA,1998-12-01,"439579,3291","4475049,263",-3.712257,40.423882
1,28079008,8,Escuelas Aguirre,Entre C/ Alcalá y C/ O’ Donell,"3°40'56.22""O","40°25'17.63""N",672,UT,Urbana tráfico,X,...,X,18900,CALLE,DE,ALCALA,1998-12-01,"442117,2366","4474770,696",-3.682316,40.421553
2,28079011,11,Ramón y Cajal,Avda. Ramón y Cajal esq. C/ Príncipe de Vergara,"3°40'38.50""O","40°27'5.29""N",709,UT,Urbana tráfico,X,...,X,610450,CALLE,DEL,PRINCIPE DE VERGARA,1998-12-01,"442564,0457","4478088,595",-3.677349,40.451473
3,28079016,16,Arturo Soria,C/ Arturo Soria esq. C/ Vizconde de los Asilos,"3°38'21.17""O","40°26'24.20""N",695,UF,Urbana fondo,X,...,None,798700,CALLE,DEL,VIZCONDE DE LOS ASILOS,1998-12-01,"445786,1729","4476796,019",-3.639242,40.440046
4,28079017,17,Villaverde,C/ Juan Peñalver,"3°42'47.89""O","40°20'49.74""N",601,UF,Urbana fondo,X,...,None,417200,CALLE,DE,JUAN PEÑALVER,1998-12-01,"439420,7015","4466532,455",-3.713317,40.347147


In [8]:
estaciones_aire.isna().sum()

CODIGO                  0
CODIGO_CORTO            0
ESTACION                0
DIRECCION               0
LONGITUD_ETRS89         0
LATITUD_ETRS89          0
ALTITUD                 0
COD_TIPO                0
NOM_TIPO                0
NO2                     0
SO2                    20
CO                     20
PM10                   11
PM2_5                  16
O3                     11
BTX                    19
COD_VIA                 0
VIA_CLASE               1
VIA_PAR                 2
VIA_NOMBRE              1
Fecha alta              0
COORDENADA_X_ETRS89     0
COORDENADA_Y_ETRS89     0
LONGITUD                0
LATITUD                 0
dtype: int64

## `distritos`

Only used as auxiliary table in the spatial join (`geometry` in ETRS89/UTM30N) -- not materialized separately in silver.

In [3]:
distritos = pd.read_parquet(DISTRITOS_PATH)
distritos.shape

(21, 7)

In [4]:
distritos.dtypes

COD_DIS        object
COD_DIS_TX     object
NOMBRE         object
DISTRI_MAY     object
DISTRI_MT      object
Area          float64
geometry       object
dtype: object

In [5]:
distritos.drop(columns="geometry").head(5)

,COD_DIS,COD_DIS_TX,NOMBRE,DISTRI_MAY,DISTRI_MT,Area
0,1,01,Centro,CENTRO,CENTRO,5.228246e+06
1,2,02,Arganzuela,ARGANZUELA,ARGANZUELA,6.462176e+06
2,3,03,Retiro,RETIRO,RETIRO,5.466211e+06
3,4,04,Salamanca,SALAMANCA,SALAMANCA,5.392404e+06
4,5,05,Chamartín,CHAMARTIN,CHAMARTÍN,9.175482e+06


In [6]:
distritos[["COD_DIS", "COD_DIS_TX", "NOMBRE", "geometry"]].head(5)

,COD_DIS,COD_DIS_TX,NOMBRE,geometry
0,1,01,Centro,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00q\x00\x0...
1,2,02,Arganzuela,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x9f\x00...
2,3,03,Retiro,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00g\x00\x0...
3,4,04,Salamanca,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00a\x00\x0...
4,5,05,Chamartín,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x96\x00...


In [10]:
from shapely import wkb

# geometry llega como WKB crudo (bytes) porque se lee con pandas, no geopandas.
# 01 = little-endian, 03000000 = tipo 3 = Polygon.
geom_decoded = distritos["geometry"].apply(wkb.loads)
geom_decoded.apply(lambda g: (g.geom_type, len(g.exterior.coords))).head(20)

0      (Polygon, 113)
1      (Polygon, 159)
2      (Polygon, 103)
3       (Polygon, 97)
4      (Polygon, 150)
5      (Polygon, 173)
6       (Polygon, 88)
7     (Polygon, 2910)
8      (Polygon, 768)
9      (Polygon, 503)
10     (Polygon, 191)
11     (Polygon, 190)
12     (Polygon, 218)
13     (Polygon, 104)
14     (Polygon, 216)
15     (Polygon, 637)
16     (Polygon, 329)
17     (Polygon, 551)
18     (Polygon, 682)
19     (Polygon, 296)
Name: geometry, dtype: object

In [7]:
distritos["geometry"].apply(type).value_counts()

geometry
<class 'bytes'>    21
Name: count, dtype: int64

In [8]:
distritos.groupby(["COD_DIS_TX", "NOMBRE"]).agg(
    n=("geometry", "size"),
    n_geometry=("geometry", "count"),
).reset_index()

,COD_DIS_TX,NOMBRE,n,n_geometry
0,01,Centro,1,1
1,02,Arganzuela,1,1
2,03,Retiro,1,1
3,04,Salamanca,1,1
4,05,Chamartín,1,1
5,06,Tetuán,1,1
6,07,Chamberí,1,1
7,08,Fuencarral - El Pardo,1,1
8,09,Moncloa - Aravaca,1,1
9,10,Latina,1,1


## `trafico_puntos_medida`

Metadata for measurement points (coordinates, type, district already included) -- feeds `dim_punto_trafico` in gold, does not pass through silver.

In [13]:
trafico_puntos = pd.read_parquet(sorted(glob.glob(TRAFICO_PUNTOS_PATH)))
trafico_puntos.shape

(9706, 9)

In [14]:
trafico_puntos.dtypes

tipo_elem     object
distrito     float64
id             int64
cod_cent      object
nombre        object
utm_x        float64
utm_y        float64
longitud     float64
latitud      float64
dtype: object

In [15]:
trafico_puntos.head(5)

,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,URB,4.0,3840,01001,Jose Ortega y Gasset E-O - Pº Castellana-Serrano,441615.343347,4.475768e+06,-3.688323,40.430502
1,URB,4.0,3841,01002,Jose Ortega y Gasset O-E - Serrano-Pº Castellana,441705.882340,4.475770e+06,-3.687256,40.430524
2,URB,1.0,3842,01003,Pº Recoletos N-S - Almirante-Prim,441319.371258,4.474841e+06,-3.691727,40.422132
3,URB,4.0,3843,01004,Pº Recoletos S-N - Pl. Cibeles- Recoletos,441301.632986,4.474764e+06,-3.691929,40.421433
4,URB,4.0,3844,01005,(AFOROS) Pº Castellana S-N - Eduardo Dato - P...,441605.765072,4.476132e+06,-3.688470,40.433782


## `trafico`

Large table -- queried directly with DuckDB over the parquet, without loading all into memory. Same schema as in silver -- cleaning is at value level (negative sentinel -> NULL), not column level.

In [16]:
duckdb.sql(f"SELECT count(*) AS n_filas FROM '{TRAFICO_PATH}'")

┌──────────┐
│ n_filas  │
│  int64   │
├──────────┤
│ 52099163 │
└──────────┘

In [2]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{TRAFICO_PATH}'")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ fecha               │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tipo_elem           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ intensidad          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ocupacion           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ carga               │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ vmed                │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ error               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ periodo_integracion │ BIGINT      │ YES     │ NULL    │ NULL  

In [18]:
duckdb.sql(f"SELECT * FROM '{TRAFICO_PATH}' LIMIT 5")

┌───────┬─────────────────────┬───────────┬────────────┬───────────┬────────┬────────┬─────────┬─────────────────────┐
│  id   │        fecha        │ tipo_elem │ intensidad │ ocupacion │ carga  │  vmed  │  error  │ periodo_integracion │
│ int64 │      timestamp      │  varchar  │   double   │  double   │ double │ double │ varchar │        int64        │
├───────┼─────────────────────┼───────────┼────────────┼───────────┼────────┼────────┼─────────┼─────────────────────┤
│  1001 │ 2023-05-01 00:00:00 │ C30       │     1152.0 │       2.0 │    0.0 │   59.0 │ N       │                   5 │
│  1001 │ 2023-05-01 00:15:00 │ C30       │      780.0 │       2.0 │    0.0 │   60.0 │ N       │                   5 │
│  1001 │ 2023-05-01 00:30:00 │ C30       │      732.0 │       2.0 │    0.0 │   63.0 │ N       │                   5 │
│  1001 │ 2023-05-01 00:45:00 │ C30       │      828.0 │       2.0 │    0.0 │   62.0 │ N       │                   5 │
│  1001 │ 2023-05-01 01:00:00 │ C30       │     

In [19]:
duckdb.sql(f"""
    SELECT
        count(*) FILTER (WHERE intensidad < 0) AS negativos_intensidad,
        count(*) FILTER (WHERE ocupacion < 0) AS negativos_ocupacion,
        count(*) FILTER (WHERE carga < 0) AS negativos_carga,
        count(*) FILTER (WHERE vmed < 0) AS negativos_vmed
    FROM '{TRAFICO_PATH}'
""")

┌──────────────────────┬─────────────────────┬─────────────────┬────────────────┐
│ negativos_intensidad │ negativos_ocupacion │ negativos_carga │ negativos_vmed │
│        int64         │        int64        │      int64      │     int64      │
├──────────────────────┼─────────────────────┼─────────────────┼────────────────┤
│                    0 │                   0 │               0 │              0 │
└──────────────────────┴─────────────────────┴─────────────────┴────────────────┘